In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm

In [2]:
# Для генерации случайных разбиений и инициализации моделей
random_seeds = [1, 42, 100, 200, 500, 1000, 2000, 3000, 4000, 5000]

In [3]:
# грузим датасет, удаляем объявления вне мкада и те, к которым внутри МКАДа не подтянулся АО
dataset = pd.read_excel('../Dataset/objects_w_geo_prepared_V0_2j.xlsx')
dataset = dataset.loc[dataset['inMKAD']==1].reset_index(drop=True)
dataset = dataset.loc[~(dataset['AO']=='НЕОПР')].reset_index(drop=True)
dataset

,Unnamed: 0,space_m2,price_per_m2,total_price,office_class,floor,maxfloor,isCU,isNonCU,isEE,...,TotalOfficeCnt_2000,classACnt_500,classACnt_1000,classACnt_2000,isMoscowCity,isSkolkovo,inMKAD,AO,inGardenRing,distanceToCenter
0,0,10.0,380,3800,ND,1,6,0,1,0,...,486,11,27,124,0,0,1,ЦАО,0,3543.300243
1,1,5.0,800,4000,A,1,8,1,0,0,...,409,0,0,313,0,0,1,ВАО,0,8554.170939
2,2,2.6,2917,7566,A,0,0,0,0,0,...,2103,1,276,703,0,0,1,САО,0,5654.790987
3,27,10.0,925,9250,A+,2,3,1,0,1,...,2281,217,120,472,0,0,1,CВАО,0,4505.076908
4,28,7.0,1357,9500,C,1,1,0,0,0,...,557,0,35,6,0,0,1,CВАО,0,7592.842639
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24327,33695,24605.0,10232,251754304,A,2,79,0,0,0,...,483,570,445,175,1,0,1,ЦАО,0,5479.273526
24328,33696,46566.3,6405,298257152,A,0,0,0,0,0,...,2241,92,56,654,0,0,1,САО,0,5366.792041
24329,33697,48297.0,6405,309342272,A,32,33,0,0,0,...,2241,92,56,654,0,0,1,САО,0,5366.792041
24330,33698,48298.0,6405,309348704,A,3,33,0,0,0,...,2241,92,56,654,0,0,1,САО,0,5366.792041


In [4]:
dataset.columns

Index(['Unnamed: 0', 'space_m2', 'price_per_m2', 'total_price', 'office_class',
       'floor', 'maxfloor', 'isCU', 'isNonCU', 'isEE', 'avg_latitude',
       'avg_longtitude', 'isMall', 'isOfficeCenter', 'inTTK', 'target',
       'minDistSubway', 'cafe_500', 'cafe_1000', 'cafe_2000', 'fast_food_500',
       'fast_food_1000', 'fast_food_2000', 'restaurant_500', 'restaurant_1000',
       'restaurant_2000', 'bank_500', 'bank_1000', 'bank_2000',
       'supermarket_500', 'supermarket_1000', 'supermarket_2000',
       'clothes_500', 'clothes_1000', 'clothes_2000', 'cosmetics_500',
       'cosmetics_1000', 'cosmetics_2000', 'electronics_500',
       'electronics_1000', 'electronics_2000', 'mall_500', 'mall_1000',
       'mall_2000', 'histplaces_500', 'histplaces_1000', 'histplaces_2000',
       'coords_str', 'TotalOfficeCnt_500', 'TotalOfficeCnt_1000',
       'TotalOfficeCnt_2000', 'classACnt_500', 'classACnt_1000',
       'classACnt_2000', 'isMoscowCity', 'isSkolkovo', 'inMKAD', 'AO',
     

In [5]:
### Размечаем уникальные здания
dataset['coords_str'] = [str(round(dataset.at[i, 'avg_latitude'], 6)) + '_' + str(round(dataset.at[i, 'avg_longtitude'], 6)) for i in dataset.index]
# число уникальных точек
print('Уникальных адресов:', len(set(dataset['coords_str'])))

l1 = sorted(list(set(dataset['coords_str'])))

address_to_num = {val: i for i, val in enumerate(l1)}
dataset['address_index'] = [address_to_num[i] for i in dataset['coords_str']]
print(min(dataset.address_index), max(dataset.address_index))
dataset

Уникальных адресов: 3080
0 3079


,Unnamed: 0,space_m2,price_per_m2,total_price,office_class,floor,maxfloor,isCU,isNonCU,isEE,...,classACnt_500,classACnt_1000,classACnt_2000,isMoscowCity,isSkolkovo,inMKAD,AO,inGardenRing,distanceToCenter,address_index
0,0,10.0,380,3800,ND,1,6,0,1,0,...,11,27,124,0,0,1,ЦАО,0,3543.300243,1694
1,1,5.0,800,4000,A,1,8,1,0,0,...,0,0,313,0,0,1,ВАО,0,8554.170939,2702
2,2,2.6,2917,7566,A,0,0,0,0,0,...,1,276,703,0,0,1,САО,0,5654.790987,2548
3,27,10.0,925,9250,A+,2,3,1,0,1,...,217,120,472,0,0,1,CВАО,0,4505.076908,2457
4,28,7.0,1357,9500,C,1,1,0,0,0,...,0,35,6,0,0,1,CВАО,0,7592.842639,2780
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24327,33695,24605.0,10232,251754304,A,2,79,0,0,0,...,570,445,175,1,0,1,ЦАО,0,5479.273526,1408
24328,33696,46566.3,6405,298257152,A,0,0,0,0,0,...,92,56,654,0,0,1,САО,0,5366.792041,2399
24329,33697,48297.0,6405,309342272,A,32,33,0,0,0,...,92,56,654,0,0,1,САО,0,5366.792041,2399
24330,33698,48298.0,6405,309348704,A,3,33,0,0,0,...,92,56,654,0,0,1,САО,0,5366.792041,2399


### Строим катбуст по 10 разным разбиениям, считаем средние метрики

In [6]:
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [7]:
# список фичей
features = [
    # В1: Помещение + здание, 
    'office_class', 'floor', 'maxfloor', 
    'isCU', 'isNonCU', 'isEE', 'isOfficeCenter', 'isMall',
    'inTTK', 'inGardenRing', 'AO',
    'space_m2',
    # В2: В1 + метро
    'minDistSubway',
    # В3: В2 + окружение
    'cafe_500', 'cafe_1000', 'cafe_2000',
    'fast_food_500', 'fast_food_1000', 'fast_food_2000', 
    'restaurant_500', 'restaurant_1000', 'restaurant_2000', 
    'bank_500', 'bank_1000', 'bank_2000', 
    'supermarket_500', 'supermarket_1000', 'supermarket_2000', 
    'clothes_500', 'clothes_1000', 'clothes_2000',
    'cosmetics_500', 'cosmetics_1000', 'cosmetics_2000',
    'electronics_500', 'electronics_1000', 'electronics_2000',
    'mall_500', 'mall_1000', 'mall_2000', 
    'histplaces_500', 'histplaces_1000', 'histplaces_2000',
    'distanceToCenter',
]
cat_features = ['office_class', 'isCU', 'isNonCU', 'isEE', 'inTTK', 'isOfficeCenter', 'isMall', 'AO']

In [8]:
mses = []
maes = []
absolutemaes = []
r2s = []

for seed in random_seeds:
    # Создаем генератор
    rng = np.random.default_rng(seed=seed)

    # берем 700 уникальных адресов
    test_index = rng.choice(max(dataset.address_index), size=700, replace=False)
    print(test_index[0:5])
    
    # делим датасет на тестовый и проверочный
    X_train = dataset.loc[~dataset['address_index'].isin(test_index)][features]
    y_train = dataset.loc[~dataset['address_index'].isin(test_index)]['target']
    X_test = dataset.loc[dataset['address_index'].isin(test_index)][features]
    y_test = dataset.loc[dataset['address_index'].isin(test_index)]['target']
    y_testRUR = dataset.loc[dataset['address_index'].isin(test_index)]['price_per_m2']
    print(y_train.shape, y_test.shape)
    
    # создаем модель
    model = CatBoostRegressor(
        iterations=400,
        learning_rate=0.015,
        random_state=seed,
        depth=10,
        loss_function='RMSE',
        cat_features = cat_features,
        verbose=0,
        l2_leaf_reg=2.0
    )
    
    # учим  модель
    model.fit(X_train, y_train)
    
    # получаем предикты
    predictions = model.predict(X_test)
    
    # MSE, MAE
    mses.append(mean_squared_error(y_test, predictions))
    maes.append(mean_absolute_error(y_test, predictions))
    
    # AbsoluteMAE
    absolutemaes.append(mean_absolute_error(np.exp(predictions), y_testRUR))
    
    # R2
    r2s.append(model.score(X_test, y_test))

[ 157   97 2862 2816 1038]
(19117,) (5215,)
[ 426 2143 2799 2861  473]
(19365,) (4967,)
[1534 1671  694 2574 2392]
(18599,) (5733,)
[1769 2564 2420 1013 2152]
(18293,) (6039,)
[2131 2427 2505 1206 2087]
(19390,) (4942,)
[1867 1891  365 2221 2546]
(18229,) (6103,)
[1316  115  666 2005 3073]
(19149,) (5183,)
[1019  243 1192  575 2093]
(18714,) (5618,)
[2907  158   17 2492 1599]
(19251,) (5081,)
[ 212  880  150 1407 1691]
(18979,) (5353,)


In [9]:
print(f'#--- AVG MSE: {np.mean(mses):.5f}' )
print(f'#--- AVG MAE: {np.mean(maes):.5f}')
print(f'#ABS AVG MAE: {np.mean(absolutemaes):.5f}')
print(f'#--- AVG  R2: {np.mean(r2s):.5f}')

#--- AVG MSE: 0.14142
#--- AVG MAE: 0.28492
#ABS AVG MAE: 1140.94945
#--- AVG  R2: 0.49626


In [10]:
### Хар-ки офиса + признаки локации
#--- AVG MSE: 0.16856
#--- AVG MAE: 0.31766
#ABS AVG MAE: 1258.41101
#--- AVG  R2: 0.39995
### -//- + метро
#--- AVG MSE: 0.15732
#--- AVG MAE: 0.30387
#ABS AVG MAE: 1202.38670
#--- AVG  R2: 0.43939
### -//- + окружение 
#--- AVG MSE: 0.14142
#--- AVG MAE: 0.28492
#ABS AVG MAE: 1140.94945
#--- AVG  R2: 0.49626

In [11]:
# Выводим значимость фичей
print("\nFeature Importances:")
dict(zip(features, model.get_feature_importance()))


Feature Importances:


{'office_class': 10.670974487111609,
 'floor': 2.72783782360377,
 'maxfloor': 8.236703194380098,
 'isCU': 0.3013439534552133,
 'isNonCU': 0.01908649200445005,
 'isEE': 0.22792447241938482,
 'isOfficeCenter': 0.7653987227831154,
 'isMall': 0.0,
 'inTTK': 0.11183567905773151,
 'inGardenRing': 0.2978570141192448,
 'AO': 1.7844650206541377,
 'space_m2': 4.371698837536603,
 'minDistSubway': 7.579401525362046,
 'cafe_500': 5.284799217793614,
 'cafe_1000': 2.142010607335765,
 'cafe_2000': 0.8409289341920974,
 'fast_food_500': 2.153575390910445,
 'fast_food_1000': 2.2095000061690553,
 'fast_food_2000': 1.0025587398273146,
 'restaurant_500': 5.84081773130588,
 'restaurant_1000': 1.1537044139766506,
 'restaurant_2000': 7.216845893814568,
 'bank_500': 1.964939643996793,
 'bank_1000': 1.010389629882617,
 'bank_2000': 0.595280932043617,
 'supermarket_500': 3.138670961181923,
 'supermarket_1000': 4.7900201699987255,
 'supermarket_2000': 3.787867210178586,
 'clothes_500': 1.8450014972763389,
 'clothe

In [12]:
# Выводим топ-7 фичей по значимости для последнего разбиения
fi_df = pd.DataFrame({
    'Feature': X_test.columns,
    'Importance': model.get_feature_importance()
})
top_15_features = fi_df.sort_values(by='Importance', ascending=False).head(7)
display(top_15_features)

,Feature,Importance
0,office_class,10.670974
2,maxfloor,8.236703
12,minDistSubway,7.579402
21,restaurant_2000,7.216846
19,restaurant_500,5.840818
13,cafe_500,5.284799
26,supermarket_1000,4.790020


### Строим линейную модель из sklearn

In [13]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [14]:
datasetLin = dataset
datasetLin.head()

,Unnamed: 0,space_m2,price_per_m2,total_price,office_class,floor,maxfloor,isCU,isNonCU,isEE,...,classACnt_500,classACnt_1000,classACnt_2000,isMoscowCity,isSkolkovo,inMKAD,AO,inGardenRing,distanceToCenter,address_index
0,0,10.0,380,3800,ND,1,6,0,1,0,...,11,27,124,0,0,1,ЦАО,0,3543.300243,1694
1,1,5.0,800,4000,A,1,8,1,0,0,...,0,0,313,0,0,1,ВАО,0,8554.170939,2702
2,2,2.6,2917,7566,A,0,0,0,0,0,...,1,276,703,0,0,1,САО,0,5654.790987,2548
3,27,10.0,925,9250,A+,2,3,1,0,1,...,217,120,472,0,0,1,CВАО,0,4505.076908,2457
4,28,7.0,1357,9500,C,1,1,0,0,0,...,0,35,6,0,0,1,CВАО,0,7592.842639,2780


In [15]:
datasetLin['nl_total_price'] = [np.log(max(0.0, i) + 1.0) for i in list(datasetLin['total_price'])]
datasetLin['nl_floor'] = [np.log(max(0.0, i) + 1.0) for i in list(datasetLin['floor'])]
datasetLin['nl_space_m2'] = [np.log(max(0.0, i) + 1.0) for i in list(datasetLin['space_m2'])]
datasetLin['nl_minDistSubway'] = [np.log(max(0.0, i) + 1.0) for i in list(datasetLin['minDistSubway'])]
datasetLin['nl_maxfloor'] = [np.log(max(0.0, i) + 1.0) for i in list(datasetLin['maxfloor'])]
env = [
    'cafe_500', 'cafe_1000', 'cafe_2000', 'fast_food_500', 'fast_food_1000', 'fast_food_2000', 'restaurant_500',
    'restaurant_1000', 'restaurant_2000', 'bank_500', 'bank_1000', 'bank_2000', 'supermarket_500', 'supermarket_1000', 'supermarket_2000', 'clothes_500', 'clothes_1000', 'clothes_2000',
    'cosmetics_500', 'cosmetics_1000', 'cosmetics_2000', 'electronics_500', 'electronics_1000', 'electronics_2000',
    'mall_500', 'mall_1000', 'mall_2000', 'histplaces_500', 'histplaces_1000', 'histplaces_2000', 'distanceToCenter'
]
for i in env:
    datasetLin['nl_' + i] = [np.log(max(0.0, i) + 1.0) for i in list(datasetLin[i])]
datasetLin = pd.get_dummies(datasetLin, columns=['office_class'], dtype=int)
datasetLin = pd.get_dummies(datasetLin, columns=['AO'], dtype=int)

linearFeatures = [
    'office_class_A', 'office_class_A+', 'office_class_B', 'office_class_B+', 'office_class_C',
    'nl_floor', 'nl_maxfloor', 'isCU', 'isNonCU', 'isEE', 'isOfficeCenter', 'isMall',
    'inTTK', 'inGardenRing',
    'AO_CВАО', 'AO_ВАО', 'AO_ЗАО', 'AO_САО', 'AO_СЗАО', 'AO_ЮАО', 'AO_ЮВАО', 'AO_ЮЗАО',
    'nl_space_m2',
    # + метро
    'nl_minDistSubway',
    # + окружение
    'nl_cafe_500', 'nl_cafe_1000', 'nl_cafe_2000',
    'nl_fast_food_500', 'nl_fast_food_1000', 'nl_fast_food_2000', 
    'nl_restaurant_500', 'nl_restaurant_1000', 'nl_restaurant_2000', 
    'nl_bank_500', 'nl_bank_1000', 'nl_bank_2000', 
    'nl_supermarket_500', 'nl_supermarket_1000', 'nl_supermarket_2000',
    'nl_clothes_500', 'nl_clothes_1000', 'nl_clothes_2000',
    'nl_cosmetics_500', 'nl_cosmetics_1000', 'nl_cosmetics_2000',
    'nl_electronics_500', 'nl_electronics_1000', 'nl_electronics_2000',
    'nl_mall_500', 'nl_mall_1000', 'nl_mall_2000',
    'nl_histplaces_500', 'nl_histplaces_1000', 'nl_histplaces_2000',
    'nl_distanceToCenter'
]
exceptFeatures = []#['total_price', 'floor', 'minDistSubway', 'maxfloor'] + env
for i in features:
    if i not in linearFeatures + exceptFeatures:
        print(i)

office_class
floor
maxfloor
AO
space_m2
minDistSubway
cafe_500
cafe_1000
cafe_2000
fast_food_500
fast_food_1000
fast_food_2000
restaurant_500
restaurant_1000
restaurant_2000
bank_500
bank_1000
bank_2000
supermarket_500
supermarket_1000
supermarket_2000
clothes_500
clothes_1000
clothes_2000
cosmetics_500
cosmetics_1000
cosmetics_2000
electronics_500
electronics_1000
electronics_2000
mall_500
mall_1000
mall_2000
histplaces_500
histplaces_1000
histplaces_2000
distanceToCenter


In [16]:
mses = []
maes = []
absolutemaes = []
r2s = []

for seed in random_seeds:
    # Создаем генератор
    rng = np.random.default_rng(seed=seed)

    # берем 700 уникальных адресов
    test_index = rng.choice(max(datasetLin.address_index), size=700, replace=False)
    print(test_index[0:5])
    
    X_train = datasetLin.loc[~datasetLin['address_index'].isin(test_index)][linearFeatures]
    y_train = datasetLin.loc[~datasetLin['address_index'].isin(test_index)]['target']
    X_test = datasetLin.loc[datasetLin['address_index'].isin(test_index)][linearFeatures]
    y_test = datasetLin.loc[datasetLin['address_index'].isin(test_index)]['target']
    y_testRUR = dataset.loc[datasetLin['address_index'].isin(test_index)]['price_per_m2']
    
    # учим модель
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    
    # считаем метрики
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    absolutemae = mean_absolute_error(np.exp(y_pred), y_testRUR)
    
    mses.append(mse)
    r2s.append(r2)
    maes.append(mae)
    absolutemaes.append(absolutemae)

[ 157   97 2862 2816 1038]
[ 426 2143 2799 2861  473]
[1534 1671  694 2574 2392]
[1769 2564 2420 1013 2152]
[2131 2427 2505 1206 2087]
[1867 1891  365 2221 2546]
[1316  115  666 2005 3073]
[1019  243 1192  575 2093]
[2907  158   17 2492 1599]
[ 212  880  150 1407 1691]


In [17]:
print(f'#--- AVG MSE: {np.mean(mses):.5f}' )
print(f'#--- AVG MAE: {np.mean(maes):.5f}')
print(f'#ABS AVG MAE: {np.mean(absolutemaes):.5f}')
print(f'#--- AVG  R2: {np.mean(r2s):.5f}')

#--- AVG MSE: 0.15843
#--- AVG MAE: 0.30771
#ABS AVG MAE: 1237.48049
#--- AVG  R2: 0.43471


In [18]:
### Хар-ки офиса + признаки локации
#--- AVG MSE: 0.17696
#--- AVG MAE: 0.32498
#ABS AVG MAE: 1271.49569
#--- AVG  R2: 0.36927
### -//- + метро
#--- AVG MSE: 0.17027
#--- AVG MAE: 0.32110
#ABS AVG MAE: 1279.64300
#--- AVG  R2: 0.39280
### -//- + окружение 
#--- AVG MSE: 0.15843
#--- AVG MAE: 0.30771
#ABS AVG MAE: 1237.48049
#--- AVG  R2: 0.43471

In [19]:
# коэфтцтенты
coefficients = model.coef_
intercept = model.intercept_

print(f"Intercept: {intercept}")
for i in range(len(coefficients)):
    if abs(coefficients[i]) > 0.12:
        print(round(coefficients[i], 5), linearFeatures[i])

Intercept: 8.593663999176224
0.25778 office_class_A
0.14212 office_class_A+
-0.1731 office_class_C
0.12357 isMall
-0.25472 AO_ВАО
-0.15029 AO_СЗАО
-0.15695 AO_ЮВАО
0.13262 nl_cafe_500
0.15816 nl_restaurant_2000
-0.15281 nl_supermarket_2000
0.13395 nl_mall_500
0.13276 nl_histplaces_500


In [20]:
coef_df = pd.DataFrame(
    {
        "Feature": X_train.columns,
        "Coefficient": model.coef_,  # Use model.coef_[0] for logistic regression
    }
)

# 2. Sort by absolute value to find the strongest positive and negative impacts
coef_df["Abs_Coefficient"] = coef_df["Coefficient"].abs()
top = coef_df.sort_values(by="Abs_Coefficient", ascending=False)

# 3. Display the result
display(top[["Feature", "Coefficient"]])

,Feature,Coefficient
0,office_class_A,0.257785
15,AO_ВАО,-0.254720
4,office_class_C,-0.173101
32,nl_restaurant_2000,0.158158
20,AO_ЮВАО,-0.156949
38,nl_supermarket_2000,-0.152806
18,AO_СЗАО,-0.150288
1,office_class_A+,0.142124
48,nl_mall_500,0.133948
51,nl_histplaces_500,0.132760


In [21]:
tempdf = pd.DataFrame()
for i in ['nl_minDistSubway', 'nl_restaurant_500', 'nl_restaurant_1000', 'nl_restaurant_2000',
         'nl_supermarket_500', 'nl_supermarket_1000', 'nl_supermarket_2000',
         'nl_fast_food_500', 'nl_fast_food_1000', 'nl_fast_food_2000',
         'nl_cafe_500', 'nl_cafe_1000', 'nl_cafe_2000',
         'nl_histplaces_500', 'nl_histplaces_1000', 'nl_histplaces_2000']:
    t = top.loc[top['Feature'] == i].reset_index(drop=True)
    tempdf = pd.concat([tempdf, t])
tempdf[['Feature', 'Coefficient']]
print(tempdf[['Coefficient']].to_string(index=False))

 Coefficient
   -0.100275
    0.051555
   -0.021027
    0.158158
   -0.041731
   -0.035723
   -0.152806
   -0.008294
   -0.051463
   -0.061644
    0.132624
    0.062706
    0.111076
    0.132760
   -0.026278
    0.042134
